In [5]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import math
from matplotlib.scale import FuncScale

from pathlib import Path

from os.path import join
import os
from functools import partial
import pathlib
import shutil

import re


In [6]:
# verif_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/mpas/mpas_fixed"
# verif_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/10m_big/forecasts"
verif_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/baselines/persistence_forecasts"


In [7]:
for init_hour in [23, 5, 11, 17]:
    init_hour = f"T{init_hour:02}"
    if init_hour:
        init_hour_name = f"{init_hour}z"
        parquets = [f for f in Path(verif_dir).iterdir() if "parquet" in f.name and f.name != "verif.parquet" and init_hour in f.name]
        eval_save_loc = join(verif_dir, f"verif_{init_hour_name}.parquet")
    else:
        parquets = [f for f in Path(verif_dir).iterdir() if "parquet" in f.name and f.name != "verif.parquet"]
        eval_save_loc = join(verif_dir, "verif.parquet")


    dfs = [pd.read_parquet(f) for f in parquets]
    # take nanmean of all verifications and save
    def nanmean_cell(*values):
        arrays = [np.atleast_1d(np.array(v, dtype=float)) for v in values]
        result = np.nanmean(arrays, axis=0)
        return result[0] if result.size == 1 else result

    df = pd.DataFrame(
        {col: [nanmean_cell(*values) for values in zip(*[df[col] for df in dfs])]
        for col in dfs[0].columns},
        index=dfs[0].index
    )
    df.attrs["init_times"] = [df.attrs["init_times"] for df in dfs]

    # df.sort(key= lambda x: x["forecast_step"]) # sort of forecast hours are in order
    df.to_parquet(eval_save_loc) # parquet keeps all the dtypes, don't have to split up np arrays in the entries
    print(f"saved verification to {eval_save_loc}")


saved verification to /glade/derecho/scratch/dkimpara/goes_10km_train/baselines/persistence_forecasts/verif_T23z.parquet
saved verification to /glade/derecho/scratch/dkimpara/goes_10km_train/baselines/persistence_forecasts/verif_T05z.parquet
saved verification to /glade/derecho/scratch/dkimpara/goes_10km_train/baselines/persistence_forecasts/verif_T11z.parquet
saved verification to /glade/derecho/scratch/dkimpara/goes_10km_train/baselines/persistence_forecasts/verif_T17z.parquet
